# Basic Decision Tree from Scratch - Classification (Gini)

In this notebook, we are going to build a decision tree from scratch using Gini as a cutoff point. 

Dataset: This is a makeup dataset that describe how much study hours a student put in and whether if they will either pass or fail their exam

| Study Hours  | Pass Exam |
| ----- | ----- |
| 1    | No    |
| 2    | No    |
| 3    | No    |
| 4    | No    |
| 5    | Yes   |
| 6    | Yes   |
| 7    | Yes   |
| 8    | Yes   |
| 9    | Yes   |
| 10    | Yes   |
| 11    | Yes   |

In [10]:
import pandas as pd
import numpy as np
import json

In [11]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [12]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Formula for Gini

### Gini Impurity Formula

For a node:

$$\text{Gini} = 1 - \sum_{k=1}^{K} p_k^2$$


Where:

- K = number of classes
- $p_k$ = proportion of class (k) in the node

---

### Binary classification (0/1 case)

$$\text{Gini} = 1 - (p_0^2 + p_1^2)$$

Where:

- $p_0 = \frac{0}{N}$
- $p_1 = \frac{1}{N}$

---

### Intuition

| Case           | Gini                          |
| -------------- | ----------------------------- |
| all same class | 0 (pure)                      |
| 50/50 split    | 0.5 (max impurity for binary) |

---

### Weighted Gini (for a split)

When you split a node into LEFT and RIGHT:

$$ \text{Weighted Gini} = \frac{N_L}{N} \cdot Gini(L) + \frac{N_R}{N} \cdot Gini(R)$$

Where:

* $N_L$ = number of samples in left node
* $N_R$ = number of samples in right node
* $N = N_L + N_R$

---

# 🧠 Intuition

Weighted Gini means:

> “how impure are the children, adjusted by their size?”

So:

* big bad child → matters a lot
* small bad child → matters less

---

# 📊 Simple example

Left:

* 4 samples → Gini = 0

Right:

* 4 samples → Gini = 0.32

$$Weighted = \frac{4}{8}(0) + \frac{4}{8}(0.32)
= 0.16$$

---

### Key insight

### Gini answers:

> “How mixed is one group?”

### Weighted Gini answers:

> “How good is this split overall?”

---

### One-line summary

```text
Gini = impurity of a node
Weighted Gini = impurity of a split
```

### Gini split procedure:
- Sort feature values
- For each adjacent pair:
    compute midpoint threshold
- For each threshold:
    split data
    compute weighted Gini
- Choose best threshold

We are going to create a tree by using Gini as a cutoff. The steps are as follows:

Build(node_data):

1. If the dataset is empty return None

2. If all labels are identical:
      create leaf and return prediction

3. If there is only one row left:
      create leaf and return prediction

4. Compute Gini and select the best weighted gini

5. Split rows into:
      left  = values <= median
      right = values > median

4. Create node storing median

5. Build(left) Recursively

6. Build(right) Recursively

In [13]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    print(mid_point)
    mid_points.append(mid_point)

1.5
2.5
3.5
4.5
5.5
6.5
7.5
8.5
9.5
10.5
11.5


In [14]:
def gini(y):
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)

    print(f"p0: {p0:.2f}, p1: {p1:.2f}")

    gini = 1 - (p0**2 + p1**2)
    return gini

In [15]:
# Test gini function
print(f"Gini impurity: {gini(df['passed']):.2f}")

p0: 0.33, p1: 0.67
Gini impurity: 0.44


In [16]:
# Split data on first mid point
split_value = mid_points[0]
df_left = df[df['studied'] <= split_value]
df_right = df[df['studied'] > split_value]

print(f"Left split:\n{df_left}\n")
print(f"Right split:\n{df_right}\n")

Left split:
   studied  passed
0        1       0

Right split:
    studied  passed
1         2       0
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1



In [17]:
# Compute Gini for the left and right splits
gini_left = gini(df_left['passed'])
gini_right = gini(df_right['passed'])

p0: 1.00, p1: 0.00
p0: 0.27, p1: 0.73


In [23]:
# weighted gini impurity
def weighted_gini(df_left, df_right):
    total_samples = len(df_left) + len(df_right)
    gini_left = gini(df_left['passed'])
    gini_right = gini(df_right['passed'])
    
    weighted_gini = (len(df_left) / total_samples) * gini_left + (len(df_right) / total_samples) * gini_right
    return weighted_gini

In [25]:
def best_gini_split(df):
    best_gini = float('inf')
    best_split_value = None
    
    for i in range(len(df['studied']) - 1):
        mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
        df_left = df[df['studied'] <= mid_point]
        df_right = df[df['studied'] > mid_point]
        
        current_gini = weighted_gini(df_left, df_right)
        
        if current_gini < best_gini:
            best_gini = current_gini
            best_split_value = mid_point
            
    return best_split_value, best_gini

In [26]:
def built_tree(df):

    if df.empty:
        return None

    # If all labels are identical, return that label
    if df['passed'].nunique() == 1:
        return int(df['passed'].iloc[0])

    # If there's only one row left, return its label
    if len(df) <= 1:
        return int(df['passed'].iloc[0])

    # Compute gini impurity of the current node
    parent_node_gini = gini(df['passed'])
    print(f"Parent node Gini impurity (reference only, not used to pick splits): {parent_node_gini:.2f}")

    # Find the best split based on Gini impurity
    median_studied, best_gini = best_gini_split(df)
    print(f"Best split value: {median_studied}, Gini after split: {best_gini:.2f}")

    # Split the dataset into two subsets based on the best split value
    left_subset = df[df['studied'] <= median_studied]
    right_subset = df[df['studied'] > median_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'median_studied': float(median_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

In [27]:
clf = built_tree(df)

p0: 0.33, p1: 0.67
Parent node Gini impurity (reference only, not used to pick splits): 0.44
p0: 1.00, p1: 0.00
p0: 0.27, p1: 0.73
p0: 1.00, p1: 0.00
p0: 0.20, p1: 0.80
p0: 1.00, p1: 0.00
p0: 0.11, p1: 0.89
p0: 1.00, p1: 0.00
p0: 0.00, p1: 1.00
p0: 0.80, p1: 0.20
p0: 0.00, p1: 1.00
p0: 0.67, p1: 0.33
p0: 0.00, p1: 1.00
p0: 0.57, p1: 0.43
p0: 0.00, p1: 1.00
p0: 0.50, p1: 0.50
p0: 0.00, p1: 1.00
p0: 0.44, p1: 0.56
p0: 0.00, p1: 1.00
p0: 0.40, p1: 0.60
p0: 0.00, p1: 1.00
p0: 0.36, p1: 0.64
p0: 0.00, p1: 1.00
Best split value: 4.5, Gini after split: 0.00
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0
3        4       0

Right subset:
    studied  passed
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1



In [28]:
clf


{'median_studied': 4.5, 'left': 0, 'right': 1}

In [29]:
# print tree structure in json format
import json
print(json.dumps(clf, indent=8))


{
        "median_studied": 4.5,
        "left": 0,
        "right": 1
}


In [30]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Median studied: {node['median_studied']}")
        print(f"{indent}├─ Left:")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right:")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Passed: {node}")


In [31]:
print_tree(clf)

    Median studied: 4.5
    ├─ Left:
    │   Passed: 0
    └─ Right:
        Passed: 1


In [32]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['median_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [33]:
prediction = predict(clf, 6)
print(f"Prediction for student who studied 5 hours: {prediction}")

Going right: 6 > 4.5
Prediction for student who studied 5 hours: 1
